In [40]:
import duckdb
import random
from functions.analyte import ANALYTES
from networks.cnn_dilated_convolutions import CNNModel
from networks.auto_encoder import AutoencoderModel
from functions.evaluation import evaluate
from functions.full_model import predict


con = duckdb.connect('../capillary.db')

df = con.execute(""" 
                 SELECT row_id, age, gender, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set
                 FROM protein_data 
                 WHERE set = 'test'
                 AND observation_nr = 1
                 """).df()

df = predict(df)
con.close()


In [41]:
import numpy as np

#temporärt kallar vi oligoklonalt för klass 3. Så klass 0 negativt, 1 positivt, 2, lätt avvikande, 3 oligoklonalt
df.loc[df['label']==4,'label'] = 3
df.loc[df['final_prediction']==4,'final_prediction'] = 3

ids = set(df.loc[(df['label'] == 0), 'row_id'].sample(50)) 
ids |= set(df.loc[(df['label'] == 1), 'row_id'].sample(50))

mat = np.zeros((4,4))
total_missclassified = sum(df['label'] != df['final_prediction'])
for i in range(4):
    for j in range(4):
        if i == j:
            continue
        mat[i,j] = (sum((df['label'] == i) & (df['final_prediction'] == j)) / total_missclassified)*97
        ids |= set(df.loc[(df['label'] == i) & (df['final_prediction'] == j),'row_id' ].sample(int(np.ceil(mat[i,j]))))
print(len(ids))

200


In [ ]:
df = df[df['row_id'].isin(ids)]
df = df.sample(frac=1).reset_index(drop=True)
print(len(df))


df = df[['row_id','value','albumin','antitrypsin','orosomukoid','haptoglobin','crp','igg','iga','igm']]
con = duckdb.connect('../application/application.db')
con.execute('DROP TABLE IF EXISTS difficult_cases')
con.execute('CREATE TABLE difficult_cases AS SELECT * FROM df')

con.close()


200
